# Assignment 4 — Decision Trees & Ensemble Methods for Credit Default

**Course:** Machine Learning (2026W)  
**Author:** Will Sutherland  
**Dataset:** [UCI Credit Card Default](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients) — 30,000 clients, binary default prediction

---

## Overview

This notebook benchmarks tree-based classifiers on an imbalanced credit risk dataset, progressing from a single decision tree through to a suite of ensemble methods.

**Key topics covered:**
- Decision tree hyperparameter analysis (`max_depth`, `min_samples_split`, `criterion`)
- Efficient hyperparameter search with `RandomizedSearchCV` (vs. exhaustive `GridSearchCV`)
- Ensemble comparison: **Random Forest**, **AdaBoost**, **Extra Trees**, **Gradient Boosted Trees**
- Learning curves to diagnose bias vs. variance
- Full head-to-head benchmark across all models using accuracy, F1, precision, and recall


## 0 · Imports & Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
)
from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV, learning_curve
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)

SEED = 42


In [ ]:
# Dataset loads directly from UCI — no local file required
DATA_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00350/"
    "default%20of%20credit%20card%20clients.xls"
)

df = pd.read_excel(DATA_URL, header=1)
df.rename(columns={"default payment next month": "DEFAULT"}, inplace=True)
df.drop(columns=["ID"], inplace=True)

print(f"Shape: {df.shape}")
print(f"\nClass distribution:\n{df['DEFAULT'].value_counts()}")
print(f"\nDefault rate: {df['DEFAULT'].mean():.3f}")
display(df.head())


In [ ]:
X = df.drop(columns=["DEFAULT"])
y = df["DEFAULT"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")


## 1 · Decision Tree — Hyperparameter Analysis

We compare four configurations to isolate the effect of three key hyperparameters:

| Config | max_depth | min_samples_split | criterion |
|---|---|---|---|
| Baseline | None | 2 | gini |
| Depth-limited | 5 | 2 | gini |
| Depth + min split | 5 | 50 | gini |
| Entropy | 5 | 2 | entropy |


In [ ]:
configs = {
    "Baseline (no limit)":       DecisionTreeClassifier(random_state=SEED),
    "max_depth=5":               DecisionTreeClassifier(max_depth=5, random_state=SEED),
    "max_depth=5, min_split=50": DecisionTreeClassifier(max_depth=5, min_samples_split=50, random_state=SEED),
    "entropy, max_depth=5":      DecisionTreeClassifier(criterion="entropy", max_depth=5, random_state=SEED),
}

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle("Decision Tree — Confusion Matrices by Configuration", fontsize=13, fontweight="bold")

print(f"{'Configuration':<30} {'Accuracy':>10} {'F1 (Default)':>14}")
print("-" * 57)

for ax, (name, clf) in zip(axes, configs.items()):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["No Default", "Default"]).plot(
        ax=ax, colorbar=False, cmap="Blues"
    )
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    ax.set_title(f"{name}\nAcc: {acc:.4f}", fontsize=9)
    print(f"{name:<30} {acc:>10.4f} {f1:>14.4f}")

plt.tight_layout()
plt.show()


**Key findings:**

1. **`max_depth` is the most impactful lever.** Removing the depth limit causes severe overfitting (~72% accuracy). Adding `max_depth=5` raises this to ~81.6% — the single biggest gain across all experiments.

2. **`min_samples_split` provides marginal additional regularization.** Combined with `max_depth=5` it produces nearly identical results, confirming both fight overfitting through different mechanisms.

3. **`criterion` (gini vs. entropy) is essentially irrelevant here** — ~0.3% difference on this dataset, with entropy being slightly slower.

4. **Class imbalance caveat:** The dataset is ~78% non-default. Overall accuracy is a misleading metric — all configurations have substantially lower recall on the default (minority) class, which matters most in credit risk.


## 2 · Randomized Hyperparameter Search

**Why `RandomizedSearchCV` over `GridSearchCV`?**

The search space defined below has 6 × 5 × 4 × 2 × 3 × 2 = **1,440 combinations**.  
`GridSearchCV` would require 1,440 × 5 folds = **7,200 model fits**.  
`RandomizedSearchCV` with `n_iter=50` runs only **250 fits** (~3% of the cost) while exploring the space efficiently.

**Why F1 as the scoring metric?**  
A naive model that predicts "no default" for everyone achieves 78% accuracy while being useless. F1 balances precision and recall on the minority class and is a much more honest measure of performance in imbalanced settings.


In [ ]:
param_dist = {
    "max_depth":         [3, 5, 7, 10, 15, None],
    "min_samples_split": [2, 10, 20, 50, 100],
    "min_samples_leaf":  [1, 5, 10, 20],
    "criterion":         ["gini", "entropy"],
    "max_features":      ["sqrt", "log2", None],
    "class_weight":      [None, "balanced"],   # addresses class imbalance
}

random_search = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=SEED),
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    random_state=SEED,
    verbose=1,
)
random_search.fit(X_train, y_train)
best_clf = random_search.best_estimator_

print(f"Best CV F1   : {random_search.best_score_:.4f}")
print(f"Best params  : {random_search.best_params_}")

y_pred_rs = best_clf.predict(X_test)
print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred_rs):.4f}")
print(f"Test F1      : {f1_score(y_test, y_pred_rs):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rs, target_names=["No Default", "Default"]))


## 3 · Ensemble Methods

We evaluate four ensemble strategies, all using 100 base estimators:

| Model | Strategy | Key Strength |
|---|---|---|
| **Random Forest** | Bagging (parallel) | Reduces variance via averaging |
| **Extra Trees** | Bagging + random splits (parallel) | Faster; more randomness reduces variance further |
| **AdaBoost** | Boosting (sequential) | Reduces bias by focusing on hard examples |
| **Gradient Boosted Trees** | Boosting (sequential, gradient) | Often best accuracy; slower to train |


In [ ]:
models = {
    "Random Forest":          RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
    "Extra Trees":            ExtraTreesClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
    "AdaBoost":               AdaBoostClassifier(n_estimators=100, random_state=SEED),
    "Gradient Boosted Trees": GradientBoostingClassifier(n_estimators=100, random_state=SEED),
}

print(f"{'Model':<26} {'Accuracy':>10} {'F1':>8} {'Precision':>11} {'Recall':>9}")
print("-" * 67)

fitted_models = {}
for name, clf in models.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    print(f"{name:<26} {acc:>10.4f} {f1:>8.4f} {prec:>11.4f} {rec:>9.4f}")
    fitted_models[name] = clf


In [ ]:
# ── Learning curves ────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 8))
gs = gridspec.GridSpec(2, 2, figure=fig)
axes = [fig.add_subplot(gs[i//2, i%2]) for i in range(4)]

for ax, (name, clf) in zip(axes, fitted_models.items()):
    train_sizes, train_scores, val_scores = learning_curve(
        clf, X_train, y_train, cv=5,
        scoring="f1", n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 8)
    )
    ax.plot(train_sizes, train_scores.mean(axis=1), label="Train F1", marker="o")
    ax.plot(train_sizes, val_scores.mean(axis=1),   label="CV F1",    marker="s")
    ax.fill_between(train_sizes,
                    train_scores.mean(axis=1) - train_scores.std(axis=1),
                    train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.1)
    ax.fill_between(train_sizes,
                    val_scores.mean(axis=1) - val_scores.std(axis=1),
                    val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.1)
    ax.set_title(name)
    ax.set_xlabel("Training Set Size")
    ax.set_ylabel("F1 Score")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("Learning Curves — Ensemble Models", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 4 · Full Benchmark — All Models

In [ ]:
all_models = {
    # Q1 — Decision Trees
    "DT — Baseline":              DecisionTreeClassifier(random_state=SEED),
    "DT — max_depth=5":          DecisionTreeClassifier(max_depth=5, random_state=SEED),
    "DT — depth=5, min_split=50":DecisionTreeClassifier(max_depth=5, min_samples_split=50, random_state=SEED),
    "DT — entropy, depth=5":     DecisionTreeClassifier(criterion="entropy", max_depth=5, random_state=SEED),
    # Q2 — RandomizedSearch best
    "DT — RandomizedSearchCV":   best_clf,
    # Q3 — Ensembles
    **fitted_models,
}

records = []
for name, clf in all_models.items():
    if name not in [m for m in fitted_models] and name != "DT — RandomizedSearchCV":
        clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    records.append({
        "Model": name,
        "Accuracy":  accuracy_score(y_test, y_pred),
        "F1":        f1_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall":    recall_score(y_test, y_pred),
    })

results_df = pd.DataFrame(records).sort_values("F1", ascending=False).reset_index(drop=True)
display(results_df.style.format({c: "{:.4f}" for c in results_df.columns if c != "Model"})
        .background_gradient(subset=["F1"], cmap="YlGn"))


## 5 · Summary & Key Takeaways

**Hyperparameter findings:**  
`max_depth` is by far the most important single lever for decision trees on this dataset. Manual tuning got most of the way there; `RandomizedSearchCV` achieved marginally better F1 by also adjusting `class_weight` and `min_samples_leaf`.

**Why ensembles win:**  
A single decision tree — no matter how well tuned — is fundamentally limited by its high variance (bagging) or high bias (heavy pruning). Ensemble methods address this directly. Random Forest and Extra Trees reduce variance through averaging; Gradient Boosted Trees reduce bias sequentially.

**Imbalanced data deserves imbalanced metrics:**  
Overall accuracy is misleading here. A model that always predicts "no default" achieves 78% accuracy. F1 on the default class is the honest scorecard, and it's the one that determines real business value in a credit risk context.

**Learning curves** show all ensemble models benefiting from more training data with no severe overfitting, suggesting additional data collection would improve performance meaningfully.
